In [0]:
# STARTER CODE - DO NOT EDIT THIS CELL

from pyspark.sql.functions import*
from pyspark.sql.types import *
from pyspark.sql import Window
from pyspark.sql.window import Window
from pyspark.sql import SparkSession

spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")


In [0]:
# STARTER CODE - DO NOT EDIT THIS CELL

#from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

customSchema = StructType([
    StructField("lpep_pickup_datetime", StringType(), True),
    StructField("lpep_dropoff_datetime", StringType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("DOLocationID", IntegerType(), True),
    StructField("passenger_count", IntegerType(), True),
    StructField("trip_distance", FloatType(), True),
    StructField("fare_amount", FloatType(), True),
    StructField("payment_type", IntegerType(), True)
])

In [0]:
# STARTER CODE - YOU CAN LOAD ANY FILE WITH A SIMILAR SYNTAX.
# Correct file path for Databricks File System (update if you are using a different volume or file name)
file_path = "/Volumes/workspace/default/q2vol/nyc-tripdata.csv"

# Read the CSV file using Spark DataFrame
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_path)

display(df.limit(5))


lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,payment_type
12/21/2018 15:17,12/21/2018 15:18,264,264,5,0.0,3.0,2
01/01/2019 0:10,01/01/2019 0:16,97,49,2,0.86,6.0,2
01/01/2019 0:27,01/01/2019 0:31,49,189,2,0.66,4.5,1
01/01/2019 0:46,01/01/2019 1:04,189,17,2,2.68,13.5,1
01/01/2019 0:19,01/01/2019 0:39,82,258,1,4.53,18.0,2


In [0]:
# LOAD THE "taxi_zone_lookup.csv" FILE SIMILARLY AS ABOVE. CAST ANY COLUMN TO APPROPRIATE DATA TYPE IF NECESSARY.

#ENTER THE CODE BELOW

zone_file_path = "/Volumes/workspace/default/q2vol/taxi_zone_lookup.csv"

taxi_zone_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(zone_file_path)

# Cast the LocationID to Integer to match PULocationID and DOLocationID in trip data
taxi_zone_df = taxi_zone_df.withColumn("LocationID", col("LocationID").cast(IntegerType()))

# Optional: preview the first 5 rows
display(taxi_zone_df.limit(5))

# =========================================================
# Filter the trip data as per assignment requirements
# Keep only rows where:
#   1. pickup and dropoff are different
#   2. trip_distance > 2.0
# =========================================================

filtered_df = df.filter(
    (col("PULocationID") != col("DOLocationID")) &
    (col("trip_distance") > 2.0)
)

# Optional: preview filtered data
display(filtered_df.limit(5))

# Optional sanity check
print("Filtered trip count:", filtered_df.count())


LocationID,Borough,Zone,service_zone
1,EWR,Newark Airport,EWR
2,Queens,Jamaica Bay,Boro Zone
3,Bronx,Allerton/Pelham Gardens,Boro Zone
4,Manhattan,Alphabet City,Yellow Zone
5,Staten Island,Arden Heights,Boro Zone


lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,payment_type
01/01/2019 0:46,01/01/2019 1:04,189,17,2,2.68,13.5,1
01/01/2019 0:19,01/01/2019 0:39,82,258,1,4.53,18.0,2
01/01/2019 0:47,01/01/2019 1:00,255,33,1,3.77,13.5,1
01/01/2019 0:12,01/01/2019 0:30,76,225,1,4.1,16.0,1
01/01/2019 0:16,01/01/2019 0:39,25,89,1,7.75,25.5,1


Filtered trip count: 305602


In [0]:
#// STARTER CODE 
# // Some commands that you can use to see your dataframes and results of the operations. You can comment the df.show(5) and uncomment display(df) to see the data differently. You will find these two functions useful in reporting your results.
df.show(5)
#display(df.limit(5))

+--------------------+---------------------+------------+------------+---------------+-------------+-----------+------------+
|lpep_pickup_datetime|lpep_dropoff_datetime|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|payment_type|
+--------------------+---------------------+------------+------------+---------------+-------------+-----------+------------+
|    12/21/2018 15:17|     12/21/2018 15:18|         264|         264|              5|          0.0|        3.0|           2|
|     01/01/2019 0:10|      01/01/2019 0:16|          97|          49|              2|         0.86|        6.0|           2|
|     01/01/2019 0:27|      01/01/2019 0:31|          49|         189|              2|         0.66|        4.5|           1|
|     01/01/2019 0:46|      01/01/2019 1:04|         189|          17|              2|         2.68|       13.5|           1|
|     01/01/2019 0:19|      01/01/2019 0:39|          82|         258|              1|         4.53|       18.0|      

In [0]:
# // STARTER CODE - DO NOT EDIT THIS CELL
# Filter the data to only keep the rows where "PULocationID" and the "DOLocationID" are different and the "trip_distance" is strictly greater than 2.0 (>2.0).

# VERY VERY IMPORTANT: ALL THE SUBSEQUENT OPERATIONS MUST BE PERFORMED ON THIS FILTERED DATA

df_filter = df.filter((df.PULocationID != df.DOLocationID) & (df.trip_distance > 2.0))
df_filter.show(5)

+--------------------+---------------------+------------+------------+---------------+-------------+-----------+------------+
|lpep_pickup_datetime|lpep_dropoff_datetime|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|payment_type|
+--------------------+---------------------+------------+------------+---------------+-------------+-----------+------------+
|     01/01/2019 0:46|      01/01/2019 1:04|         189|          17|              2|         2.68|       13.5|           1|
|     01/01/2019 0:19|      01/01/2019 0:39|          82|         258|              1|         4.53|       18.0|           2|
|     01/01/2019 0:47|      01/01/2019 1:00|         255|          33|              1|         3.77|       13.5|           1|
|     01/01/2019 0:12|      01/01/2019 0:30|          76|         225|              1|          4.1|       16.0|           1|
|     01/01/2019 0:16|      01/01/2019 0:39|          25|          89|              1|         7.75|       25.5|      

In [0]:
from pyspark.sql.functions import col, desc, count

# PART 1a: Top-5 most popular drop locations
# Count drop-offs per DOLocationID
top5_dropoff = (
    df_filter.groupBy("DOLocationID")
             .agg(count("*").alias("number_of_dropoffs"))
             .orderBy(desc("number_of_dropoffs"), col("DOLocationID").asc())  # tie-breaker: lower DOLocationID
             .limit(5)
)

# Ensure correct data types
top5_dropoff = top5_dropoff.select(
    col("DOLocationID").cast("int"),
    col("number_of_dropoffs").cast("int")
)

# Display the results
top5_dropoff.show()





+------------+------------------+
|DOLocationID|number_of_dropoffs|
+------------+------------------+
|          61|              5937|
|         138|              5146|
|         239|              4133|
|         244|              4006|
|          42|              3859|
+------------+------------------+



In [0]:
from pyspark.sql.functions import col, desc, count

# PART 1b: Top-5 most popular pickup locations
# Count pickups per PULocationID
top5_pickup = (
    df_filter.groupBy("PULocationID")
             .agg(count("*").alias("number_of_pickups"))
             .orderBy(desc("number_of_pickups"), col("PULocationID").asc())  # tie-breaker: lower PULocationID
             .limit(5)
)

# Ensure correct data types
top5_pickup = top5_pickup.select(
    col("PULocationID").cast("int"),
    col("number_of_pickups").cast("int")
)

# Display the results
top5_pickup.show()



+------------+-----------------+
|PULocationID|number_of_pickups|
+------------+-----------------+
|          74|            17360|
|          75|            13299|
|         244|             9958|
|          41|             9645|
|          82|             9306|
+------------+-----------------+



In [0]:
# PART 2: List the top-3 locations with the maximum overall activity, i.e. sum of all pickups and all dropoffs at that LocationID. In case of a tie, the lower LocationID gets listed first.

# Output Schema: LocationID int, number_activities int

# Hint: In order to get the result, you may need to perform a join operation between the two dataframes that you created in earlier parts (to come up with the sum of the number of pickups and dropoffs on each location). 

# ENTER THE CODE BELOW
from pyspark.sql.functions import col, coalesce, lit, desc

# Step 1: Count pickups per location
pickup_counts = df_filter.groupBy("PULocationID") \
                         .count() \
                         .withColumnRenamed("count", "number_of_pickups")

# Step 2: Count dropoffs per location
dropoff_counts = df_filter.groupBy("DOLocationID") \
                          .count() \
                          .withColumnRenamed("count", "number_of_dropoffs")

# Step 3: Outer join on LocationID
overall_activity = pickup_counts.join(
    dropoff_counts,
    pickup_counts.PULocationID == dropoff_counts.DOLocationID,
    how="outer"
).select(
    coalesce(col("PULocationID"), col("DOLocationID")).alias("LocationID"),
    coalesce(col("number_of_pickups"), lit(0)).alias("number_of_pickups"),
    coalesce(col("number_of_dropoffs"), lit(0)).alias("number_of_dropoffs")
)

# Step 4: Compute total activities
overall_activity = overall_activity.withColumn(
    "total_activity", col("number_of_pickups") + col("number_of_dropoffs")
)

# Step 5: Sort descending by total_activity, then ascending by LocationID
top3_activity = overall_activity.orderBy(desc("total_activity"), col("LocationID").asc()).limit(3)

# Display results
top3_activity.show()


+----------+-----------------+------------------+--------------+
|LocationID|number_of_pickups|number_of_dropoffs|total_activity|
+----------+-----------------+------------------+--------------+
|        74|            17360|              2932|         20292|
|        75|            13299|              3027|         16326|
|       244|             9958|              4006|         13964|
+----------+-----------------+------------------+--------------+



In [0]:
# PART 3: List all the boroughs (including "Unknown" and "EWR") in the order of having the highest to lowest number of activities (i.e. sum of all pickups and all dropoffs at that LocationID), along with the total number of activity counts for each borough in NYC during that entire period of time.

# Output Schema: Borough string, total_number_activities int

# Hint: You can use the dataframe obtained from the previous part, and will need to do the join with the 'taxi_zone_lookup' dataframe. Also, checkout the "agg" function applied to a grouped dataframe.

# ENTER THE CODE BELOW
# PART 3: Total activity per borough
from pyspark.sql.functions import col, sum as spark_sum

# Ensure overall_activity has all locations (outer join pickups & dropoffs)
overall_activity = pickup_counts.join(dropoff_counts,
                                      pickup_counts.PULocationID == dropoff_counts.DOLocationID,
                                      how='outer') \
                                .withColumn("LocationID", col("PULocationID")) \
                                .fillna({"number_of_pickups": 0, "number_of_dropoffs": 0}) \
                                .withColumn("number_activities", col("number_of_pickups") + col("number_of_dropoffs"))

# Join with taxi_zone_df to get borough
activity_with_borough = overall_activity.join(
    taxi_zone_df,
    overall_activity.LocationID == taxi_zone_df.LocationID,
    how="left"
)

# Replace null Borough with 'NULL'
activity_with_borough = activity_with_borough.fillna({"Borough": "NULL"})

# Group by Borough and sum total activities
borough_activity_sorted = activity_with_borough.groupBy("Borough") \
                                               .agg(spark_sum("number_activities").alias("total_number_activities")) \
                                               .orderBy(col("total_number_activities").desc())

borough_activity_sorted.show()


+-------------+-----------------------+
|      Borough|total_number_activities|
+-------------+-----------------------+
|     Brooklyn|                 198506|
|    Manhattan|                 175953|
|       Queens|                 157633|
|        Bronx|                  67707|
|         NULL|                   9198|
|      Unknown|                   1215|
|Staten Island|                    888|
|          EWR|                    104|
+-------------+-----------------------+



In [0]:
# PART 4: List the top 2 days of week with the largest number of daily average pickups, along with the average number of pickups on each of the 2 days in descending order (no rounding off required). Here, the average pickup is calculated by taking an average of the number of pick-ups on different dates falling on the same day of the week. For example, 02/01/2021, 02/08/2021 and 02/15/2021 are all Mondays, so the average pick-ups for these is the sum of the pickups on each date divided by 3.

# Note: The day of week is a string of the day’s full spelling, e.g., "Monday" instead of the		number 1 or "Mon". Also, the pickup_datetime is in the format: yyyy-mm-dd.

# Output Schema: day_of_week string, avg_count float

# Hint: You may need to group by the "date" (without time stamp - time in the day) first. Checkout "date_format" and "to_date" functions.

# ENTER THE CODE BELOW

from pyspark.sql.functions import to_date, date_format, col, avg, count

# Step 1: Extract just the date from pickup datetime
df_dates = df_filter.withColumn("pickup_date", to_date(col("lpep_pickup_datetime"), "MM/dd/yyyy HH:mm"))

# Step 2: Count pickups per date
daily_pickups = df_dates.groupBy("pickup_date").agg(count("*").alias("num_pickups"))

# Step 3: Add day of the week as full string
daily_pickups = daily_pickups.withColumn("day_of_week", date_format(col("pickup_date"), "EEEE"))

# Step 4: Compute average pickups per day of the week
avg_pickups_by_day = daily_pickups.groupBy("day_of_week") \
                                  .agg(avg("num_pickups").alias("avg_count")) \
                                  .orderBy(col("avg_count").desc())

# Step 5: Select top 2 days
top2_days = avg_pickups_by_day.limit(2)

# Display results
top2_days.show()


+-----------+---------+
|day_of_week|avg_count|
+-----------+---------+
|  Wednesday|  10257.6|
|   Saturday|  9884.75|
+-----------+---------+



In [0]:
# PART 5: For each particular hour of a day (0 to 23, 0 being midnight) - in their order from 0 to 23, find the zone in Brooklyn borough with the LARGEST number of pickups. 

# Note: All dates for each hour should be included.

# Output Schema: hour_of_day int, zone string, max_count int

# Hint: You may need to use "Window" over hour of day, along with "group by" to find the MAXIMUM count of pickups

# ENTER THE CODE BELOW

from pyspark.sql.functions import hour, to_timestamp, col, count, row_number
from pyspark.sql.window import Window

# Extract hour of day using correct format
df_hours = df_filter.withColumn(
    "hour_of_day",
    hour(to_timestamp(col("lpep_pickup_datetime"), "MM/dd/yyyy HH:mm"))
)

# Join with taxi_zone_df to get borough info
df_brooklyn = df_hours.join(
    taxi_zone_df,
    df_hours.PULocationID == taxi_zone_df.LocationID,
    how="left"
).filter(col("Borough") == "Brooklyn")

# Count pickups per hour per zone
hourly_zone_counts = df_brooklyn.groupBy("hour_of_day", "Zone") \
                                .agg(count("*").alias("pickup_count"))

# Use window to get top zone per hour
window_spec = Window.partitionBy("hour_of_day").orderBy(col("pickup_count").desc(), col("Zone").asc())

top_zone_per_hour = hourly_zone_counts.withColumn("rank", row_number().over(window_spec)) \
                                      .filter(col("rank") == 1) \
                                      .select(
                                          col("hour_of_day"),
                                          col("Zone").alias("zone"),
                                          col("pickup_count").alias("max_count")
                                      ) \
                                      .orderBy("hour_of_day")

top_zone_per_hour.show(24)



+-----------+--------------------+---------+
|hour_of_day|                zone|max_count|
+-----------+--------------------+---------+
|          0|Williamsburg (Nor...|      569|
|          1|Williamsburg (Nor...|      460|
|          2|Williamsburg (Nor...|      429|
|          3|Williamsburg (Nor...|      357|
|          4|Williamsburg (Nor...|      228|
|          5|   East Williamsburg|       89|
|          6|       East New York|      149|
|          7|    Brooklyn Heights|      307|
|          8|    Brooklyn Heights|      511|
|          9|    Brooklyn Heights|      574|
|         10|    Brooklyn Heights|      502|
|         11|    Brooklyn Heights|      563|
|         12|    Brooklyn Heights|      491|
|         13|    Brooklyn Heights|      472|
|         14|         Fort Greene|      511|
|         15|         Fort Greene|      559|
|         16|         Fort Greene|      658|
|         17|Downtown Brooklyn...|      651|
|         18|         Fort Greene|      736|
|         

In [0]:
# PART 6 - Find which 3 different days in the month of January, in Manhattan, saw the largest positive percentage increase in pick-ups compared to the previous day, in the order from largest percentage increase to smallest percentage increase 

# Note: All years need to be aggregated to calculate the pickups for a specific day of January. The change from Dec 31 to Jan 1 can be excluded.

# Output Schema: day int, percent_change float

# Hint: You might need to use lag function, over a window ordered by day of month.

# ENTER THE CODE BELOW

from pyspark.sql.functions import to_date, dayofmonth, col, count, lag
from pyspark.sql.window import Window

# Step 1: Extract pickup date and day of month
df_manhattan = df_filter.join(
    taxi_zone_df,
    df_filter.PULocationID == taxi_zone_df.LocationID,
    how="left"
).filter(col("Borough") == "Manhattan")

df_manhattan = df_manhattan.withColumn("pickup_date", to_date(col("lpep_pickup_datetime"), "MM/dd/yyyy HH:mm")) \
                           .withColumn("day", dayofmonth(col("pickup_date")))

# Step 2: Aggregate pickups per day in January (all years)
january_pickups = df_manhattan.filter(month(col("pickup_date")) == 1) \
                              .groupBy("day") \
                              .agg(count("*").alias("num_pickups")) \
                              .orderBy("day")

# Step 3: Use lag to get previous day's pickups
window_spec = Window.orderBy("day")
january_pickups = january_pickups.withColumn("prev_day_pickups", lag("num_pickups").over(window_spec))

# Step 4: Calculate percentage change compared to previous day
january_pickups = january_pickups.withColumn(
    "percent_change",
    ((col("num_pickups") - col("prev_day_pickups")) / col("prev_day_pickups")) * 100
)

# Step 5: Keep only positive changes and select top 3
top3_jan_increase = january_pickups.filter(col("percent_change") > 0) \
                                   .orderBy(col("percent_change").desc()) \
                                   .select("day", "percent_change") \
                                   .limit(3)

# Display results
top3_jan_increase.show()




/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1061: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---+------------------+
|day|    percent_change|
+---+------------------+
| 22| 51.06260769672601|
|  2|  28.3776144714528|
| 28|22.939560439560438|
+---+------------------+

